# Historical Projection Sensitivity Lab

## A standings-geometry teaching experiment

**Question.** If one fixed projected player-season stat vector were added on top of each comparable completed historical team-season, how would that team's exact category ranks and roto points have changed within the observed league table?

This is an **additive historical perturbation**. It adds production without removing a rostered player, enforcing roster slots, changing opponents, applying draft costs, or changing any future outcome. The local top-30 projection snapshot supplies illustrative vectors; it is not a player board. Select one example at a time and study the mathematics.

## The calculation

For completed season $s$, target historical team $i$, other team $j$, selected projected player $p$, category $c$, exposure $\lambda\in[0,1]$, projected games $G_p$, and category weight $w_c$, effective games are $g_p(\lambda)=\lambda G_p$.

For a counting category with projected per-game rate $a_{pc}$, the additive contribution is $d_{pc}(\lambda)=g_p(\lambda)a_{pc}$. Only the target changes: $x'_{sic}=x_{sic}+d_{pc}$. For a percentage, production is added as makes and attempts: $x'_{sic}=(M_{sic}+m_{pc})/(A_{sic}+a_{pc})$. Percentages are never added directly.

After orienting lower-is-better categories by $y=-x$ and all others by $y=x$, exact average-tie rank is $r=1+\#(y_j>y_i)+\#(j\ne i, y_j=y_i)/2$. Roto points are $q=(n_s-r+1)w_c$, and the local table effect is $\Delta q=q'-q$. The bundle $B=\sum_c\Delta q$ is called a **total roto-point perturbation**, never player value.

In [ ]:
from IPython.display import display

import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go

from analysis.projection_sensitivity import (
    PROJECTION_STAT_BASIS,
    contribution_rows,
    effect_summary,
    exposure_response,
    historical_effects,
    load_sensitivity_lab,
    scenario_bundle,
    scenario_category,
)

pn.extension("plotly", "tabulator")
# This declaration is a hard gate, not a guess. The notebook fails closed if changed.
assert PROJECTION_STAT_BASIS == "per_game"


## 1. Synthetic counting example: tiers and average ties

Suppose a category has values A=100, B=90, C=90, D=70. A is first; B and C share second and third, so each receives rank 2.5; D is fourth. With four teams and unit weight, the corresponding points are 4, 2.5, 2.5, and 1. Adding 25 to D makes D=95: D crosses the 90 tier, moving to second and gaining 2 roto points. The execution below makes the tie rule inspectable.

In [ ]:
def average_tie_rank(values, target_index):
    target = values[target_index]
    return 1 + sum(value > target for value in values) + sum(
        index != target_index and value == target for index, value in enumerate(values)
    ) / 2

values = [100, 90, 90, 70]
before_rank = average_tie_rank(values, 3)
after_rank = average_tie_rank([100, 90, 90, 95], 3)
before_points, after_points = 5 - before_rank, 5 - after_rank
pd.DataFrame({"team": list("ABCD"), "value before": values,
              "rank before": [average_tie_rank(values, i) for i in range(4)],
              "D points gained after +25": [None, None, None, after_points - before_points]})


## 2. Synthetic ratio example: volume matters

If D made 460 of 1,000 attempts, its field-goal percentage is .460. A player at 10 makes on 20 attempts per game over 20 effective games contributes 200 makes and 400 attempts. The combined percentage is $(460+200)/(1000+400)=.4714$. Writing $.460+.500$ would be invalid because percentages are rates with different denominators.

In [ ]:
team_makes, team_attempts = 460, 1000
player_makes_pg, player_attempts_pg, effective_games = 10, 20, 20
combined = (team_makes + player_makes_pg * effective_games) / (team_attempts + player_attempts_pg * effective_games)
pd.DataFrame([{"team FG%": team_makes / team_attempts,
               "player FG%": player_makes_pg / player_attempts_pg,
               "volume-aware combined FG%": combined,
               "invalid direct addition": team_makes / team_attempts + player_makes_pg / player_attempts_pg}])


## 3. Evidence gates

A historical season enters only when its completed roto settings and team evidence are usable, all teams have every included category, categories match the newest completed reference exactly, percentage makes/attempts reconcile to the stored rate and archived points, the final-period horizon is within 5% of the reference, and there are at least six teams. One common complete-case population is used for the eight-category bundle.

The snapshot enters only when it is the latest local 2026-27 public top-30 capture, its row count reconciles, and all needed projected primitives are finite and internally consistent. No missing primitive becomes zero. The metadata below intentionally excludes raw player rows.

The source displays percentage and makes/attempts at different rounding resolutions, so the displayed components need not divide back to the displayed percentage exactly. This notebook uses the displayed percentage as the projected rate and displayed attempts as the projected volume. It derives an internally coherent expected-makes value as percentage × attempts. Displayed makes remain audit evidence. This deterministic reconciliation does not recover or claim hidden provider precision.

In [ ]:
# The helper resolves the repository root from this notebook directory.
lab = load_sensitivity_lab()
display(pd.DataFrame([{"historical status": lab.historical.status, "historical conclusion": lab.historical.conclusion,
                       "projection status": lab.projections.status, "projection conclusion": lab.projections.conclusion,
                       "declared projection basis": PROJECTION_STAT_BASIS}]))
display(lab.historical.audit)
display(lab.projections.audit)


## 4. Interactive lab

The four tabs follow one calculation: inspect what is added; inspect one historical table; repeat that same perturbation over every eligible target team-season; then see category effects and the predeclared exposure grid. Season-balanced medians give each season equal weight after taking its within-season team median. Positive, zero, and negative rates are empirical historical frequencies, not probabilities or confidence intervals.

At exposure zero, the original table must reproduce exactly. As exposure increases, ranks can change only when a value crosses a historical tier, so the summary is naturally step-shaped.

In [ ]:
if lab.status != "COMPLETE":
    display(pn.pane.Alert(f"{lab.status}: the lab intentionally stops.", alert_type="warning"))
else:
    tables = lab.historical.tables
    players = tuple(sorted(lab.projections.snapshot.players, key=lambda row: row.source_display_name.casefold()))
    player_select = pn.widgets.Select(name="Projected-player example", options={row.source_display_name: row for row in players})
    exposure = pn.widgets.FloatSlider(name="Exposure λ", start=0, end=1, step=.05, value=1)
    season_select = pn.widgets.Select(name="Historical season", options={str(row.season): row for row in tables})
    category_select = pn.widgets.Select(name="Scored category", options=[row.code for row in tables[0].categories])
    team_select = pn.widgets.Select(name="Target historical team")

    def set_team_options(event=None):
        current = season_select.value
        options = {row.team: row.team_id for _, row in current.teams.drop_duplicates("team_id").sort_values("team").iterrows()}
        team_select.options = options
        team_select.value = next(iter(options.values()))

    season_select.param.watch(set_team_options, "value")
    set_team_options()

    def rank_ladder(table, player, target, category, lambda_):
        rows = scenario_category(table, player, target, category, lambda_)
        figure = go.Figure()
        for _, row in rows.iterrows():
            color = "#b74a3c" if row.is_target else "#526777"
            figure.add_trace(go.Scatter(x=[row.before_rank, row.after_rank], y=[row.team, row.team], mode="lines+markers",
                line={"color": color}, marker={"size": 9}, name=row.team, showlegend=False,
                hovertemplate=(f"{row.team}<br>before: value={row.before_value:.4g}, rank={row.before_rank:.2f}, points={row.before_points:.2f}"
                               f"<br>after: value={row.after_value:.4g}, rank={row.after_rank:.2f}, points={row.after_points:.2f}<extra></extra>")))
        figure.update_layout(title=f"Observed {table.season} {category} table: before → after", xaxis_title="Average-tie rank (1 is best)", yaxis_title="Historical team")
        figure.update_xaxes(autorange="reversed")
        return figure

    def render(player, lambda_, table, target, category):
        contribution = contribution_rows(player, table.categories, lambda_)
        one_table = scenario_category(table, player, target, category, lambda_)
        effects = historical_effects(tables, player, lambda_)
        summary = effect_summary(effects)
        variation = px.strip(effects, x="delta_points", y="category", facet_col="season", stripmode="overlay",
                             title="Every eligible target team-season: exact Δ roto points")
        variation.add_vline(x=0, line_dash="dot", line_color="black")
        bundle = px.strip(effects, x="delta_points", y="category", stripmode="overlay",
                          title="Selected example: category-wise historical perturbations")
        response = exposure_response(tables, player)
        curve = px.line(response, x="exposure", y="season_balanced_median", line_shape="hv",
                        title="Exposure response: season-balanced median bundle perturbation")
        curve.add_scatter(x=response.exposure, y=response.q1, mode="lines", line={"dash": "dot"}, name="season-median Q1")
        curve.add_scatter(x=response.exposure, y=response.q3, mode="lines", line={"dash": "dot"}, name="season-median Q3")
        contribution_display = contribution.rename(columns={"displayed_rate": "Displayed rate", "displayed_attempts_per_game": "Displayed attempts/game", "derived_effective_makes_per_game": "Derived effective makes/game", "displayed_makes_per_game_audit_only": "Displayed makes/game (audit only)", "displayed_vs_derived_makes_difference": "Displayed-vs-derived makes difference"})
        return pn.Tabs(
            ("1 · Contribution math", pn.Column(pn.pane.Markdown("Rates × effective games, or derived makes/attempts for ratios."), pn.widgets.Tabulator(contribution_display, disabled=True))),
            ("2 · One league table", pn.Column(pn.pane.Plotly(rank_ladder(table, player, target, category, lambda_), config={"responsive": True}), pn.widgets.Tabulator(one_table, disabled=True))),
            ("3 · Historical variation", pn.Column(pn.pane.Plotly(variation, config={"responsive": True}), pn.widgets.Tabulator(summary, disabled=True))),
            ("4 · Category bundle", pn.Column(pn.pane.Plotly(bundle, config={"responsive": True}), pn.pane.Plotly(curve, config={"responsive": True}))),
        )

    controls = pn.Row(player_select, exposure, season_select, team_select, category_select)
    dashboard = pn.Column(controls, pn.bind(render, player_select, exposure, season_select, team_select, category_select))
    # A conditional block does not automatically display its final expression in Jupyter.
    # Explicit display makes this notebook-first lab visible in JupyterLab and Panel Preview.
    display(dashboard)


## Interpretation boundary

A positive perturbation means only that this fixed vector would have improved some observed historical table under the deliberately unrealistic additive rule. It does not say the player was available, replaceable, affordable, healthier, more certain, or more desirable than another player. The top 30 is a selected provider subset, and the experiment makes no claim about the rest of a future player pool.

These results show how fixed projected stat vectors would have perturbed observed historical league tables. They do not estimate draft value, scarcity, replacement value, future standings, or recommended selections.